# EDA & Yesterday Construction

In [1]:
import pandas as pd
df = pd.read_csv('hydro_data.csv')
print(len(df))
print(df.head())
print(df.columns)

623546
     Datetime Hydro ID      mean    target  Hydro Lat  Hydro Long
0  2017-12-31  07FA004  2.271110  0.153475   56.19933   -120.8145
1  2018-01-01  07FA004  2.190741  0.089377   56.19933   -120.8145
2  2018-01-02  07FA004  2.219304  0.108082   56.19933   -120.8145
3  2018-01-03  07FA004  1.991847 -0.072357   56.19933   -120.8145
4  2018-01-04  07FA004  1.761454 -0.258988   56.19933   -120.8145
Index(['Datetime', 'Hydro ID', 'mean', 'target', 'Hydro Lat', 'Hydro Long'], dtype='object')


In [2]:
df['Datetime'] = pd.to_datetime(df['Datetime'])

In [3]:
# Create a yesterday column for merging
df['yesterday'] = df['Datetime'] - pd.Timedelta(days=1)

# Create the new mean_prev column by merging the dataframe with itself
df = df.merge(df[['Datetime', 'Hydro ID', 'mean']], 
               left_on=['yesterday', 'Hydro ID'], 
               right_on=['Datetime', 'Hydro ID'],
               how='left',
               suffixes=('', '_prev')
)
# Drop the temporary column
df.drop(columns=['Datetime_prev'], inplace=True)


In [4]:
print(len(df))
print(df.head())
print(df.columns)

623546
    Datetime Hydro ID      mean    target  Hydro Lat  Hydro Long  yesterday  \
0 2017-12-31  07FA004  2.271110  0.153475   56.19933   -120.8145 2017-12-30   
1 2018-01-01  07FA004  2.190741  0.089377   56.19933   -120.8145 2017-12-31   
2 2018-01-02  07FA004  2.219304  0.108082   56.19933   -120.8145 2018-01-01   
3 2018-01-03  07FA004  1.991847 -0.072357   56.19933   -120.8145 2018-01-02   
4 2018-01-04  07FA004  1.761454 -0.258988   56.19933   -120.8145 2018-01-03   

   mean_prev  
0        NaN  
1   2.271110  
2   2.190741  
3   2.219304  
4   1.991847  
Index(['Datetime', 'Hydro ID', 'mean', 'target', 'Hydro Lat', 'Hydro Long',
       'yesterday', 'mean_prev'],
      dtype='object')


# Joining Climate ID to hydro_data

In [5]:
climate_df = pd.read_csv('Hydro_to_Climate_Mapping.csv')
print(len(climate_df))
print(climate_df.head())
print(climate_df.columns)

237
  Climate ID               Climate Station  Climate Lat  Climate Long  \
0    1125079                   MERRITT STP        50.11       -120.80   
1    1085836  OOTSA LAKESKINS LAKE CLIMATE        53.77       -126.00   
2    1021989           COURTENAY PUNTLEDGE        49.69       -125.03   
3    1012055                 LAKE COWICHAN        48.83       -124.05   
4    1173220                GOLDEN AIRPORT        51.30       -116.98   

                               Climate Geometry Hydro ID  \
0   POINT (1371584.753754437 579673.5698140148)  08LG010   
1             POINT (1000000 973941.8603673474)  08JA023   
2  POINT (1070015.0536907623 519789.9835525174)  08HB006   
3  POINT (1143370.3057236315 425729.8146816146)  08HA002   
4   POINT (1626558.1740378956 738469.657341446)  08NA006   

                              Hydro Station  Hydro Lat  Hydro Long  \
0                COLDWATER RIVER AT MERRITT   50.10980  -120.80300   
1  NECHAKO RESERVOIR AT SKINS LAKE SPILLWAY   53.77238  

In [6]:
df = df.merge(climate_df[['Hydro ID', 'Climate ID']], on='Hydro ID', how='left')

In [7]:
print(len(df))
print(df.head())
print(df.columns)

623546
    Datetime Hydro ID      mean    target  Hydro Lat  Hydro Long  yesterday  \
0 2017-12-31  07FA004  2.271110  0.153475   56.19933   -120.8145 2017-12-30   
1 2018-01-01  07FA004  2.190741  0.089377   56.19933   -120.8145 2017-12-31   
2 2018-01-02  07FA004  2.219304  0.108082   56.19933   -120.8145 2018-01-01   
3 2018-01-03  07FA004  1.991847 -0.072357   56.19933   -120.8145 2018-01-02   
4 2018-01-04  07FA004  1.761454 -0.258988   56.19933   -120.8145 2018-01-03   

   mean_prev Climate ID  
0        NaN    1183001  
1   2.271110    1183001  
2   2.190741    1183001  
3   2.219304    1183001  
4   1.991847    1183001  
Index(['Datetime', 'Hydro ID', 'mean', 'target', 'Hydro Lat', 'Hydro Long',
       'yesterday', 'mean_prev', 'Climate ID'],
      dtype='object')


# Adding Day Of and Previous Day's Precipitation to hydro_data

## Creating Climate DF

In [8]:
import glob

climate_files = glob.glob('climate-data/*.csv')
climate_list = []

for f in climate_files:
    c_df = pd.read_csv(f, usecols=['Date/Time', 'Climate ID', 'Total Precip (mm)'], encoding='latin-1')
    c_df['Date/Time'] = pd.to_datetime(c_df['Date/Time'])
    climate_list.append(c_df)
full_climate_df = pd.concat(climate_list)
full_climate_df.rename(columns={'Date/Time': 'Datetime'}, inplace=True)
full_climate_df['yesterday'] = full_climate_df['Datetime'] - pd.Timedelta(days=1)

In [9]:
print(len(full_climate_df))
print(full_climate_df.head())
print(full_climate_df.columns)

383118
  Climate ID   Datetime  Total Precip (mm)  yesterday
0    1012055 2018-01-01                NaN 2017-12-31
1    1012055 2018-01-02                NaN 2018-01-01
2    1012055 2018-01-03                NaN 2018-01-02
3    1012055 2018-01-04                9.0 2018-01-03
4    1012055 2018-01-05                NaN 2018-01-04
Index(['Climate ID', 'Datetime', 'Total Precip (mm)', 'yesterday'], dtype='object')


## Joining Climate and Original DF

In [10]:
# This removes decimals (.0), removes spaces, and makes everything a string
df['Climate ID'] = df['Climate ID'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
full_climate_df['Climate ID'] = full_climate_df['Climate ID'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Add today's precipitation to the main dataframe
df = df.merge(full_climate_df[['Datetime', 'Climate ID', 'Total Precip (mm)']], 
               on=['Datetime', 'Climate ID'],
               how='left',
               suffixes=('', '_today')
)
# Add yesterday's precipitation to the main dataframe
df = df.merge(full_climate_df[['Datetime', 'Climate ID', 'Total Precip (mm)']],
            left_on=['yesterday', 'Climate ID'],
            right_on=['Datetime', 'Climate ID'],
            how='left',
            suffixes=('', '_yesterday')
)
df.drop(columns=['Datetime_yesterday'], inplace=True)

# Debugging

In [11]:
print(len(df))
print(df.head())
print(df.columns)
# Calculate percentages
nan_pct = (df[['Total Precip (mm)', 'Total Precip (mm)_yesterday']].isna().mean() * 100).round(2)

print("--- Missing Data Percentage ---")
for col, pct in nan_pct.items():
    print(f"{col}: {pct}% missing")

623546
    Datetime Hydro ID      mean    target  Hydro Lat  Hydro Long  yesterday  \
0 2017-12-31  07FA004  2.271110  0.153475   56.19933   -120.8145 2017-12-30   
1 2018-01-01  07FA004  2.190741  0.089377   56.19933   -120.8145 2017-12-31   
2 2018-01-02  07FA004  2.219304  0.108082   56.19933   -120.8145 2018-01-01   
3 2018-01-03  07FA004  1.991847 -0.072357   56.19933   -120.8145 2018-01-02   
4 2018-01-04  07FA004  1.761454 -0.258988   56.19933   -120.8145 2018-01-03   

   mean_prev Climate ID  Total Precip (mm)  Total Precip (mm)_yesterday  
0        NaN    1183001                NaN                          NaN  
1   2.271110    1183001                0.0                          NaN  
2   2.190741    1183001                0.0                          0.0  
3   2.219304    1183001                0.0                          0.0  
4   1.991847    1183001                0.0                          0.0  
Index(['Datetime', 'Hydro ID', 'mean', 'target', 'Hydro Lat', 'Hydro Long'

In [12]:
# Find which stations are missing the most data
missing_by_station = df.groupby('Climate ID')['Total Precip (mm)'].apply(lambda x: x.isna().mean() * 100)
print("\n--- Top 5 Stations with Missing Data ---")
print(missing_by_station.sort_values(ascending=False).head(15))
# This will put the 0.03% stations at the top
print(missing_by_station.sort_values(ascending=True).head(15))


--- Top 5 Stations with Missing Data ---
Climate ID
nan        100.000000
1181513    100.000000
116Q20D     74.224962
1161650     52.198991
1125079     52.191732
1012055     50.810697
1125073     45.907341
1114619     44.945806
1024638     41.927541
1101571     41.116573
1154203     40.621651
1125766     40.348630
1095018     40.200000
1073615     39.275415
1090660     37.392857
Name: Total Precip (mm), dtype: float64
Climate ID
1015628    0.035174
1113581    0.089726
10253G0    0.176699
1145460    0.214133
1105192    0.462963
1173210    0.526685
1031316    0.633468
1100031    0.673639
1126070    0.715728
1021830    0.806734
1046391    1.122413
114B1F0    1.147561
1168520    1.506245
1176755    1.553124
1097646    1.577169
Name: Total Precip (mm), dtype: float64


In [13]:
target_id = '1181513'     

# Filter and look at the raw data
preview = full_climate_df[full_climate_df['Climate ID'] == target_id]

print(f"--- Data Preview for Station {target_id} ---")
print(preview.head(10))

# Check if there are ANY non-null precipitation values for this station
non_null_count = preview['Total Precip (mm)'].notna().sum()
print(f"\nTotal records for this station: {len(preview)}")
print(f"Records with actual precipitation data: {non_null_count}")

--- Data Preview for Station 1181513 ---
  Climate ID   Datetime  Total Precip (mm)  yesterday
0    1181513 2018-01-01                NaN 2017-12-31
1    1181513 2018-01-02                NaN 2018-01-01
2    1181513 2018-01-03                NaN 2018-01-02
3    1181513 2018-01-04                NaN 2018-01-03
4    1181513 2018-01-05                NaN 2018-01-04
5    1181513 2018-01-06                NaN 2018-01-05
6    1181513 2018-01-07                NaN 2018-01-06
7    1181513 2018-01-08                NaN 2018-01-07
8    1181513 2018-01-09                NaN 2018-01-08
9    1181513 2018-01-10                NaN 2018-01-09

Total records for this station: 3287
Records with actual precipitation data: 0


# Checking Which Climate Stations Have NO Recorded Precipitation Information

In [14]:
# Group by Climate ID and check if ALL values are NaN
all_nan_stations = df.groupby('Climate ID')['Total Precip (mm)'].apply(lambda x: x.isna().all())

# Filter for only the True values
dead_stations = all_nan_stations[all_nan_stations].index.tolist()

print(f"Count of stations with 100% missing data: {len(dead_stations)}")
print("Sample of Dead Stations:", dead_stations[:10])

Count of stations with 100% missing data: 2
Sample of Dead Stations: ['1181513', 'nan']


# Checking Which Climate Stations (From our Mapping) Have no Corresponding Climate Files

In [15]:
for station in dead_stations[:10]:  # Just show the first 10 for brevity
    preview = full_climate_df[full_climate_df['Climate ID'] == station]

    # print(f"--- Data Preview for Station {station} ---")
    # print(preview.head(10))

    # Check if there are ANY non-null precipitation values for this station
    non_null_count = preview['Total Precip (mm)'].notna().sum()
    if len(preview) == 0:
        print(f"No records found for this {station}.")
    # print(f"\nTotal records for this station: {len(preview)}")
    # print(f"Records with actual precipitation data: {non_null_count}")

No records found for this nan.


# Deleting Stations With No Precipitation Data or Climate Files

In [16]:
print(f"Original unique stations: {df['Climate ID'].nunique()}")
# Find the IDs that have 100% NaNs for today's precip
missing_stats = df.groupby('Climate ID')['Total Precip (mm)'].apply(lambda x: x.isna().all())
dead_ids = missing_stats[missing_stats].index.tolist()

print(f"Dropping {len(dead_ids)} stations with zero data...")

# Filter the main dataframe to KEEP only rows NOT in the dead_ids list
df = df[~df['Climate ID'].isin(dead_ids)].copy()

# Final verification
print(f"New row count: {len(df)}")
print(f"Remaining unique stations: {df['Climate ID'].nunique()}")

Original unique stations: 103
Dropping 2 stations with zero data...
New row count: 606847
Remaining unique stations: 101


# Replacing Missing Precipitation Values With 0

In [ ]:
# Create a dictionary to specify which columns to fill
fill_values = {'Total Precip (mm)': 0.0, 'Total Precip (mm)_yesterday': 0.0, 'mean_prev': 0.0}

# Apply the fill
df.fillna(value=fill_values, inplace=True)

# Verification
print("Missing values after fill:")
print(df[['Total Precip (mm)', 'Total Precip (mm)_yesterday', 'mean_prev']].isna().sum())

Missing values after fill:
Total Precip (mm)              0
Total Precip (mm)_yesterday    0
mean_prev                      0
dtype: int64
Missing values after fill:
mean         0
mean_prev    0
dtype: int64


In [18]:
print(df.head())
print(df.columns)

    Datetime Hydro ID      mean    target  Hydro Lat  Hydro Long  yesterday  \
0 2017-12-31  07FA004  2.271110  0.153475   56.19933   -120.8145 2017-12-30   
1 2018-01-01  07FA004  2.190741  0.089377   56.19933   -120.8145 2017-12-31   
2 2018-01-02  07FA004  2.219304  0.108082   56.19933   -120.8145 2018-01-01   
3 2018-01-03  07FA004  1.991847 -0.072357   56.19933   -120.8145 2018-01-02   
4 2018-01-04  07FA004  1.761454 -0.258988   56.19933   -120.8145 2018-01-03   

   mean_prev Climate ID  Total Precip (mm)  Total Precip (mm)_yesterday  
0   0.000000    1183001                0.0                          0.0  
1   2.271110    1183001                0.0                          0.0  
2   2.190741    1183001                0.0                          0.0  
3   2.219304    1183001                0.0                          0.0  
4   1.991847    1183001                0.0                          0.0  
Index(['Datetime', 'Hydro ID', 'mean', 'target', 'Hydro Lat', 'Hydro Long',
     

In [21]:
rename_map = {
    'mean_prev': 'mean_yesterday',
    'Total Precip (mm)': 'precip',
    'Total Precip (mm)_yesterday': 'precip_yesterday'
}
df_final = df.rename(columns=rename_map)
cols_to_keep = [
    'Datetime', 
    'Hydro ID',
    'mean', 
    'mean_yesterday', 
    'precip', 
    'precip_yesterday', 
    'target'
]
df_final = df_final[cols_to_keep]
df_output = df_final.copy()
print(df_output.head())
print(df_output.shape)
df_output.to_csv('final_hydro_climate_data.csv', index=False, compression='gzip')

    Datetime Hydro ID      mean  mean_yesterday  precip  precip_yesterday  \
0 2017-12-31  07FA004  2.271110        0.000000     0.0               0.0   
1 2018-01-01  07FA004  2.190741        2.271110     0.0               0.0   
2 2018-01-02  07FA004  2.219304        2.190741     0.0               0.0   
3 2018-01-03  07FA004  1.991847        2.219304     0.0               0.0   
4 2018-01-04  07FA004  1.761454        1.991847     0.0               0.0   

     target  
0  0.153475  
1  0.089377  
2  0.108082  
3 -0.072357  
4 -0.258988  
(606847, 7)
